# Deep Learning Lab: MNIST on GPU

This notebook follows the repository's numbered Lab 1 style and runs the reusable implementation in `mnist_gpu_lab.py`. It covers a CNN, the Faster R-CNN MNIST annotation limitation, VGG16/AlexNet transfer learning, and a ViT built from scratch.

**Runtime:** Google Colab or Kaggle with GPU enabled. The script downloads MNIST at runtime and does not commit datasets or checkpoints.

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'torchvision', 'scikit-learn'])

In [ ]:
from pathlib import Path
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# If this notebook is opened from GitHub, run the script from the repository root.
from importlib.util import spec_from_file_location, module_from_spec
script = Path('Part1_MNIST_Deep_Learning/mnist_gpu_lab.py')
spec = spec_from_file_location('mnist_gpu_lab', script)
lab = module_from_spec(spec)
spec.loader.exec_module(lab)
train_loader, test_loader = lab.loaders('./data', batch_size=128, limit=12000)
transfer_train_loader, transfer_test_loader = lab.transfer_loaders('./data', batch_size=32, limit=12000)
print('Device:', lab.device())


## FCNN, CNN and ViT classification

The smoke configuration below is intentionally short. Increase `epochs` and remove `limit` for the final GPU run.

In [ ]:
results = {}
for name, model in [('FCNN', lab.FCNN()), ('CNN', lab.CNN()), ('ViT', lab.ViT())]:
    results[name] = lab.train(model, train_loader, test_loader, epochs=2, lr=1e-3)
    print(name, results[name])

## VGG16 and AlexNet fine tuning

For a strict ImageNet-transfer experiment, use `weights=models.VGG16_Weights.DEFAULT` or the corresponding AlexNet weights and a three-channel 224x224 transform. The compact script keeps the smoke run offline-friendly.

In [ ]:
for name in ['vgg16', 'alexnet']:
    results[name] = lab.train(lab.transfer_model(name), transfer_train_loader, transfer_test_loader, epochs=1, lr=1e-4)
    print(name, results[name])

In [ ]:
import pandas as pd
comparison = pd.DataFrame([
    {'Model': name, 'Loss': value.loss, 'Accuracy': value.accuracy,
     'Macro F1': value.macro_f1, 'Training time (s)': value.seconds}
    for name, value in results.items()
])
comparison.sort_values('Accuracy', ascending=False)

## Faster R-CNN protocol

MNIST is a classification dataset and has no bounding boxes. The detector constructor is included for a fair architecture demonstration, but a real detector experiment must generate or annotate meaningful boxes. The helper below creates a synthetic full-image box only to show the expected Torchvision target schema; detection should be reported with mAP/IoU rather than classification accuracy alone.

In [ ]:
detector = lab.faster_rcnn_for_mnist()
image, digit = train_loader.dataset[0]
target = lab.mnist_detection_target(image, int(digit))
print(type(detector).__name__, target)
detector_results = lab.train_faster_rcnn(detector, lab.detection_loader('./data', batch_size=8, limit=256), epochs=1)
results['Faster R-CNN (synthetic boxes)'] = detector_results
print('Faster R-CNN (synthetic boxes)', detector_results)

## Comparison and synthesis

Compare `loss`, `accuracy`, `macro_f1`, and `seconds` from `results`. On native 28x28 MNIST, CNN is normally the best accuracy/compute baseline. ViT demonstrates patch-token modeling but needs careful tuning. Large ImageNet backbones add compute and resizing overhead. The Faster R-CNN result uses synthetic full-image boxes, so it is a protocol demonstration rather than a genuine detection benchmark.

The lab's main lesson is to match architecture and evaluation metrics to both the task objective and annotation format.